In [2]:
import pandas as pd

sales_data = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "store": ["Calgary", "Calgary", "Edmonton", "Edmonton", "Vancouver", "Vancouver", "Toronto", "Toronto"],
    "category": ["Electronics", "Furniture", "Electronics", "Office Supplies", "Furniture", "Electronics", "Office Supplies", "Furniture"],
    "revenue": [2500, 1800, 3200, 900, 2200, 4100, 750, 2800],
    "cost": [1700, 1200, 2300, 500, 1600, 2900, 400, 2100],
    "units_sold": [5, 3, 8, 10, 4, 7, 9, 5],
})
sales_data

,order_id,store,category,revenue,cost,units_sold
0,101,Calgary,Electronics,2500,1700,5
1,102,Calgary,Furniture,1800,1200,3
2,103,Edmonton,Electronics,3200,2300,8
3,104,Edmonton,Office Supplies,900,500,10
4,105,Vancouver,Furniture,2200,1600,4
5,106,Vancouver,Electronics,4100,2900,7
6,107,Toronto,Office Supplies,750,400,9
7,108,Toronto,Furniture,2800,2100,5


In [9]:
def calculate_profit(revenue, cost):
    return revenue - cost

print("Q1:", calculate_profit(2500, 1700))

Q1: 800


In [10]:
def calculate_margin(revenue, cost):
    if revenue <= 0:
        return None
    return (revenue - cost) / revenue

print("Q2:", calculate_margin(2500, 1700))

Q2: 0.32


In [11]:
def add_profit_and_margin(df):
    df = df.copy()
    df["profit"] = df.apply(lambda r: calculate_profit(r["revenue"], r["cost"]), axis=1)
    df["margin"] = df.apply(lambda r: calculate_margin(r["revenue"], r["cost"]), axis=1)
    return df

df = add_profit_and_margin(sales_data)
df

,order_id,store,category,revenue,cost,units_sold,profit,margin
0,101,Calgary,Electronics,2500,1700,5,800,0.320000
1,102,Calgary,Furniture,1800,1200,3,600,0.333333
2,103,Edmonton,Electronics,3200,2300,8,900,0.281250
3,104,Edmonton,Office Supplies,900,500,10,400,0.444444
4,105,Vancouver,Furniture,2200,1600,4,600,0.272727
5,106,Vancouver,Electronics,4100,2900,7,1200,0.292683
6,107,Toronto,Office Supplies,750,400,9,350,0.466667
7,108,Toronto,Furniture,2800,2100,5,700,0.250000


In [12]:
def classify_margin(margin):
    if margin >= 0.35:
        return "Strong"
    elif margin >= 0.20:
        return "Average"
    else:
        return "Weak"

print(classify_margin(0.40), classify_margin(0.25), classify_margin(0.10))

Strong Average Weak


In [13]:
df["margin_level"] = df["margin"].apply(classify_margin)
df

,order_id,store,category,revenue,cost,units_sold,profit,margin,margin_level
0,101,Calgary,Electronics,2500,1700,5,800,0.320000,Average
1,102,Calgary,Furniture,1800,1200,3,600,0.333333,Average
2,103,Edmonton,Electronics,3200,2300,8,900,0.281250,Average
3,104,Edmonton,Office Supplies,900,500,10,400,0.444444,Strong
4,105,Vancouver,Furniture,2200,1600,4,600,0.272727,Average
5,106,Vancouver,Electronics,4100,2900,7,1200,0.292683,Average
6,107,Toronto,Office Supplies,750,400,9,350,0.466667,Strong
7,108,Toronto,Furniture,2800,2100,5,700,0.250000,Average


In [14]:
def summarize_sales_by_store(df):
    summary = df.groupby("store").agg(
        total_revenue=("revenue", "sum"),
        total_cost=("cost", "sum"),
        total_units=("units_sold", "sum"),
    ).reset_index()
    summary["total_profit"] = summary["total_revenue"] - summary["total_cost"]
    summary = summary[["store", "total_revenue", "total_cost", "total_profit", "total_units"]]
    return summary.sort_values("total_profit", ascending=False).reset_index(drop=True)

store_summary = summarize_sales_by_store(df)
order_counts = df.groupby("store")["order_id"].count().reset_index(name="num_orders")
store_summary = store_summary.merge(order_counts, on="store")
store_summary

,store,total_revenue,total_cost,total_profit,total_units,num_orders
0,Vancouver,6300,4500,1800,11,2
1,Calgary,4300,2900,1400,8,2
2,Edmonton,4100,2800,1300,18,2
3,Toronto,3550,2500,1050,14,2


In [15]:
def find_top_store(store_summary):
    top = store_summary.loc[store_summary["total_profit"].idxmax()]
    return {"store": top["store"], "total_profit": top["total_profit"]}

find_top_store(store_summary)

{'store': 'Vancouver', 'total_profit': np.int64(1800)}

In [16]:
def calculate_average_order_value(total_revenue, num_orders):
    if num_orders == 0:
        return 0
    return total_revenue / num_orders

store_summary["avg_order_value"] = store_summary.apply(
    lambda r: calculate_average_order_value(r["total_revenue"], r["num_orders"]), axis=1)
store_summary

,store,total_revenue,total_cost,total_profit,total_units,num_orders,avg_order_value
0,Vancouver,6300,4500,1800,11,2,3150.0
1,Calgary,4300,2900,1400,8,2,2150.0
2,Edmonton,4100,2800,1300,18,2,2050.0
3,Toronto,3550,2500,1050,14,2,1775.0


In [17]:
def flag_store_performance(total_profit):
    if total_profit >= 1500:
        return "High Performer"
    elif total_profit >= 800:
        return "Stable"
    else:
        return "Needs Review"

store_summary["flag"] = store_summary["total_profit"].apply(flag_store_performance)
store_summary

,store,total_revenue,total_cost,total_profit,total_units,num_orders,avg_order_value,flag
0,Vancouver,6300,4500,1800,11,2,3150.0,High Performer
1,Calgary,4300,2900,1400,8,2,2150.0,Stable
2,Edmonton,4100,2800,1300,18,2,2050.0,Stable
3,Toronto,3550,2500,1050,14,2,1775.0,Stable
